In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython
import math
from IPython.display import Image, Math

plt.rcParams["figure.figsize"] = (14,4);
plt.gray();

# the curse of dimensionality

In $N$-dimensional space, the Euclidean distance loses much of its meaning since the relative distances between uniformly spaced points become all the same.

In [ ]:
dims = range(2,50)
e_ratios = []
c_avgs = []
for N in dims:
    # Generate 1000 random points in N dimensions.
    P = [np.random.randint(-100, 100, N) for _ in range(10000)]
    # reference point
    Q = np.random.randint(-100,100,N)
    q_norm = np.linalg.norm(Q)
    # euclidean distances
    e_dist = [np.linalg.norm(p - Q) for p in P]
    e_ratios.append( np.log10(max(e_dist) / (min(e_dist) + 0.0001)) )
    # cosine distance
    c_dist = [np.inner(p, Q) / np.linalg.norm(p) / q_norm  for p in P]
    c_avgs.append(np.mean(c_dist))

plt.plot(dims, e_ratios, label='max dist / min dist (log scale)')
plt.plot(dims, c_avgs, label='avg cos dist')
plt.xlabel('Number of dimensions');
plt.legend();

# The Haar basis

In [ ]:
def haar1D(M, k='all', normalize=False):
    N = 2 ** M
    
    def haar_vector(n):
        if n == 0:
            h, k = np.ones(N), N
        else:
            h = np.zeros(N)
            # express n > 0 as 2^p + q with p as large as possible;
            # then k = N/2^p is the length of the support and s = qk is the shift
            p = math.floor(math.log(n) / math.log(2))
            pp = int(pow(2, p))
            k = int(N / pp)
            s = int((n - pp) * k)
            h[s:s+k//2] = 1
            h[s+k//2:s+k] = -1
        return h  / (np.sqrt(k) if normalize else 1)

    if k == 'all':
        return np.array([haar_vector(k) for k in range(N)])
    else:
        return haar_vector(k)

In [ ]:
def show_haar(M: int):
    COLS = 4
    basis = haar1D(M)
    rows = int(np.ceil(len(basis) / COLS))
    for n in range(len(basis)):
        plt.subplot(rows, COLS, n+1)
        plt.stem(basis[n])
    plt.tight_layout(pad=2)

In [ ]:
show_haar(3)

### check for orthogonality

In [ ]:
basis = haar1D(3)

# since the Haar basis is real, we simply need to transpose without conjugation
print(np.allclose(basis @ basis.T, np.eye(len(basis))))
print(np.allclose(basis.T @ basis, np.eye(len(basis))))

that's because the vectors were not normalized, let's redo it

In [ ]:
basis = haar1D(3, normalize=True)

# since the Haar basis is real, we simply need to transpose without conjugation
print(np.allclose(basis @ basis.T, np.eye(len(basis))))
print(np.allclose(basis.T @ basis, np.eye(len(basis))))

## Haar basis expansion

In [ ]:
# Let's define some test signals of length 64
M = 6
N = 2 ** M

test_signal = [np.zeros(N), np.zeros(N)]
# first test signal is a box sequence
test_signal[0][(N//4):(N//4 + N//2)] = 1

# second one a sinusoid that completes 2 periods over the length of the signal
test_signal[1] = np.sin(4 * np.pi * np.arange(0, N) / N)

for n, sig in enumerate(test_signal):
    plt.subplot(1, 2, n+1)
    plt.plot(test_signal[n])

In [ ]:
def haar_decomposition(x : np.ndarray) -> np.ndarray:
    # make sure the length of the input vector is a power of two
    N = len(x)
    M = int(np.log(N) / np.log(2))
    assert 2 ** M == N, 'input length must be a power of two'
    return haar1D(M, normalize=True) @ x

In [ ]:
def haar_reconstruction(w : np.ndarray) -> np.ndarray:
    N = len(w)
    M = int(np.log(N) / np.log(2))
    assert 2 ** M == N, 'input length must be a power of two'
    return haar1D(M, normalize=True).T @ w

In [ ]:
plt.figure(figsize=(10, 5))
for n, sig in enumerate(test_signal):
    plt.subplot(2, 3, 3*n+1)
    plt.plot(sig)
    plt.title('original signal (canonical basis)')    
    hc = haar_decomposition(sig)
    plt.subplot(2, 3, 3*n+2)
    plt.stem(hc)    
    plt.title('coefficients in the Haar basis')
    plt.subplot(2, 3, 3*n+3)
    plt.plot(haar_reconstruction(hc))
    plt.title('reconstruction using Haar basis vectors')
plt.tight_layout()

## Robustness of the Haar representation

In [ ]:
def compress(x, k):
    hc = haar_decomposition(x)
    idx = np.argsort(np.abs(hc))[::-1][:k]
    coeff_subset = np.zeros(len(x))
    coeff_subset[idx[:k]] = hc[idx[:k]]
    return haar_reconstruction(coeff_subset)

In [ ]:
P = 4
for p in range(1, P+1):
    plt.subplot(P // 4, 4, p)
    plt.plot(compress(test_signal[0], p))
    plt.title(f'{p} largest coefficients')
plt.tight_layout()

For the sinusoidal signal we need more coefficients:

In [ ]:
P = 12
for p in range(1, P+1):
    plt.subplot(P // 4, 4, p)
    k = 1 if p == 1 else (p - 1) * 5
    plt.plot(compress(test_signal[1], k))
    plt.title(f'{p} largest coefficients')
plt.tight_layout()

## 2D Haar basis for images

In [ ]:
def multishow(*images):
    fig, axes = plt.subplots(nrows=1, ncols=len(images))
    for i, s in enumerate(images):
        axes[i].matshow(s);
    plt.show()

In [ ]:
def haar2D(M, k):
    N = 2 ** M
    hr = haar1D(M, k % N)
    hv = haar1D(M, k // N)
    # 2D Haar basis matrix is separable, so we can just take the column-row product
    H = np.outer(hr, hv)
    return H / math.sqrt(np.sum(H * H))

In [ ]:
multishow(
    haar2D(3, 1), 
    haar2D(3, 2), 
    haar2D(3, 3), 
    haar2D(3, 62), 
    haar2D(3, 63),
);

In [ ]:
cameraman = np.array(plt.imread('img/cameraman.jpg'), dtype=int)
plt.matshow(cameraman);

In [ ]:
# Project the image onto the Haar basis, obtaining a vector of 4096 coefficients.
# This is simply the analysis formula for the vector space with an orthogonal basis.
M = 6
N = 2 ** M
tx_img = np.zeros(N * N)
for k in range(N * N):
    tx_img[k] = np.sum(cameraman * haar2D(M, k))

# Now rebuild the image with the synthesis formula; since the basis is orthonormal.
# We just need to scale the basis matrices by the projection coefficients.
rx_img = np.zeros((N, N))
for k in range(N * N):
    rx_img += tx_img[k] * haar2D(M, k)

multishow(tx_img.reshape(N, N), rx_img);

## Transmission outage when using the Haar decomposition

In [ ]:
# oops, we lose half the data
lossy_img = np.copy(tx_img);
lossy_img[int(len(tx_img)/2):] = 0

# rebuild matrix
rx_img = np.zeros((N, N))
for k in range(N * N):
    rx_img += lossy_img[k] * haar2D(M, k)

plt.matshow(rx_img);


Note that if we lose the first half of the coefficients the result is markedly different:

In [ ]:
lossy_img = np.copy(tx_img);
lossy_img[:int(len(tx_img)/2)] = 0

rx_img = np.zeros((N, N))
for k in range(N * N):
    rx_img += lossy_img[k] * haar2D(M, k)

plt.matshow(rx_img);